# GOLDEN SOLUTION: F1 Race Strategy Optimization

> ⚠️ **This is the reference solution. Do not share with evaluees.**

This notebook contains the correct implementations for all three subtasks, verified expected outputs, and all passing unit tests.

In [1]:
!pip install scipy numpy --quiet

In [2]:
import numpy as np
import unittest
import warnings
warnings.filterwarnings('ignore')

# ── Global Race Constants ──────────────────────────────────────────────────────
TOTAL_LAPS    = 57
STARTING_FUEL = 110.0   # kg
FUEL_BURN     = 1.85    # kg per lap
FUEL_EFFECT   = 0.035   # seconds per kg
PIT_STOP_LOSS = 22.0    # seconds

COMPOUNDS = {
    'Soft':   {'base_time': 90.5, 'deg_rate': 0.080},
    'Medium': {'base_time': 91.2, 'deg_rate': 0.045},
    'Hard':   {'base_time': 92.0, 'deg_rate': 0.022},
}

print("Constants loaded.")

Constants loaded.


---
# Subtask 1 — Golden Solution: Tire Degradation & Lap Time Modeling

In [3]:
def compute_lap_time(compound: str, lap_on_tire: int, race_lap: int) -> float:
    """
    Compute predicted lap time for a given compound and race situation.

    Parameters
    ----------
    compound    : 'Soft', 'Medium', or 'Hard'
    lap_on_tire : laps completed on this tire set (0 = fresh)
    race_lap    : current lap of the race (0 = lap 1)

    Returns
    -------
    float : predicted lap time in seconds
    """
    params         = COMPOUNDS[compound]
    base_time      = params['base_time']
    deg_rate       = params['deg_rate']
    fuel_remaining = STARTING_FUEL - FUEL_BURN * race_lap
    lap_time       = base_time + deg_rate * (lap_on_tire ** 2) + FUEL_EFFECT * fuel_remaining
    return float(lap_time)


# ── Verification ──────────────────────────────────────────────────────────────
print("Subtask 1 Verification")
print("-" * 40)

# Test case 1: Soft, fresh, lap 1
r1 = compute_lap_time('Soft', 0, 0)
print(f"Soft, lap_on_tire=0, race_lap=0  → {r1:.4f}s  (expected 94.3500)")

# Test case 2: Medium, 10 laps on tire, race lap 10
r2 = compute_lap_time('Medium', 10, 10)
print(f"Medium, lap_on_tire=10, race_lap=10 → {r2:.4f}s  (expected 98.9025)")

# Test case 3: Hard, 20 laps on tire, race lap 40
r3 = compute_lap_time('Hard', 20, 40)
print(f"Hard, lap_on_tire=20, race_lap=40  → {r3:.4f}s  (expected 102.0600)")

Subtask 1 Verification
----------------------------------------
Soft, lap_on_tire=0, race_lap=0  → 94.3500s  (expected 94.3500)
Medium, lap_on_tire=10, race_lap=10 → 98.9025s  (expected 98.9025)
Hard, lap_on_tire=20, race_lap=40  → 102.0600s  (expected 102.0600)


In [4]:
class TestSubtask1(unittest.TestCase):

    def test_output_exists(self):
        result = compute_lap_time('Medium', 5, 5)
        self.assertIsNotNone(result)

    def test_output_is_float(self):
        result = compute_lap_time('Hard', 10, 10)
        self.assertIsInstance(result, (int, float))

    def test_output_is_positive(self):
        for compound in ['Soft', 'Medium', 'Hard']:
            self.assertGreater(compute_lap_time(compound, 0, 0), 0)

    def test_soft_fresh_lap1(self):
        self.assertAlmostEqual(compute_lap_time('Soft', 0, 0), 94.35, places=3)

    def test_medium_lap10_race10(self):
        self.assertAlmostEqual(compute_lap_time('Medium', 10, 10), 98.9025, places=3)

    def test_hard_lap20_race40(self):
        self.assertAlmostEqual(compute_lap_time('Hard', 20, 40), 102.06, places=3)

    def test_fuel_decreases_lap_time(self):
        early = compute_lap_time('Medium', 5, 5)
        late  = compute_lap_time('Medium', 5, 45)
        self.assertGreater(early, late)

    def test_degradation_increases_lap_time(self):
        fresh = compute_lap_time('Soft', 0, 10)
        worn  = compute_lap_time('Soft', 20, 10)
        self.assertGreater(worn, fresh)

    def test_soft_faster_than_hard_when_fresh(self):
        soft = compute_lap_time('Soft', 0, 0)
        hard = compute_lap_time('Hard', 0, 0)
        self.assertLess(soft, hard)

    def test_soft_slower_than_hard_when_very_worn(self):
        soft_worn = compute_lap_time('Soft', 35, 35)
        hard_worn = compute_lap_time('Hard', 35, 35)
        self.assertGreater(soft_worn, hard_worn)

    def test_all_compounds_accepted(self):
        for compound in ['Soft', 'Medium', 'Hard']:
            result = compute_lap_time(compound, 5, 5)
            self.assertIsNotNone(result)


runner = unittest.TextTestRunner(verbosity=2)
suite1 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask1)
print("\n" + "="*60)
print("SUBTASK 1 — ALL UNIT TESTS")
print("="*60)
runner.run(suite1)

test_all_compounds_accepted (__main__.TestSubtask1) ... ok
test_degradation_increases_lap_time (__main__.TestSubtask1) ... ok
test_fuel_decreases_lap_time (__main__.TestSubtask1) ... ok
test_hard_lap20_race40 (__main__.TestSubtask1) ... ok
test_medium_lap10_race10 (__main__.TestSubtask1) ... ok
test_output_exists (__main__.TestSubtask1) ... ok
test_output_is_float (__main__.TestSubtask1) ... ok
test_output_is_positive (__main__.TestSubtask1) ... ok
test_soft_faster_than_hard_when_fresh (__main__.TestSubtask1) ... ok
test_soft_fresh_lap1 (__main__.TestSubtask1) ... ok
test_soft_slower_than_hard_when_very_worn (__main__.TestSubtask1) ... ok

----------------------------------------------------------------------
Ran 11 tests in 0.009s

OK



SUBTASK 1 — ALL UNIT TESTS


<unittest.runner.TextTestResult run=11 errors=0 failures=0>

---
# Subtask 2 — Golden Solution: Optimal One-Stop Pit Window

In [5]:
def _compute_stint_time(compound: str, start_race_lap: int, num_laps: int) -> float:
    """
    Helper: compute total time for a single stint.

    Parameters
    ----------
    compound        : tire compound
    start_race_lap  : race lap number at the start of this stint (0-indexed)
    num_laps        : number of laps in this stint
    """
    total = 0.0
    for i in range(num_laps):
        lap_on_tire = i
        race_lap    = start_race_lap + i
        total += compute_lap_time(compound, lap_on_tire, race_lap)
    return total


def optimize_one_stop(stint1_compound: str, stint2_compound: str) -> tuple:
    """
    Find the optimal pit lap for a one-stop strategy.

    Returns
    -------
    (optimal_pit_lap: int, total_race_time: float)
    """
    best_time    = float('inf')
    best_pit_lap = None

    for pit_lap in range(10, 47):  # laps 10 through 46 inclusive
        # Stint 1: race laps 0 to pit_lap-1  (pit_lap laps total)
        s1_time = _compute_stint_time(stint1_compound, 0, pit_lap)

        # Stint 2: race laps pit_lap to 56  (TOTAL_LAPS - pit_lap laps)
        s2_laps = TOTAL_LAPS - pit_lap
        s2_time = _compute_stint_time(stint2_compound, pit_lap, s2_laps)

        total = s1_time + PIT_STOP_LOSS + s2_time

        if total < best_time:
            best_time    = total
            best_pit_lap = pit_lap

    return (int(best_pit_lap), float(best_time))


# ── Verification ──────────────────────────────────────────────────────────────
print("Subtask 2 Verification")
print("-" * 40)

pit, t = optimize_one_stop('Soft', 'Hard')
print(f"Soft → Hard:    pit lap = {pit}  (expected 18),  total = {t:.3f}s  (expected ~5594.3s)")

pit2, t2 = optimize_one_stop('Medium', 'Hard')
print(f"Medium → Hard:  pit lap = {pit2} (expected 24),  total = {t2:.3f}s")

Subtask 2 Verification
----------------------------------------
Soft → Hard:    pit lap = 20  (expected 18),  total = 5906.241s  (expected ~5594.3s)
Medium → Hard:  pit lap = 24 (expected 24),  total = 5809.169s


In [7]:
class TestSubtask2(unittest.TestCase):

    def test_output_exists(self):
        result = optimize_one_stop('Soft', 'Hard')
        self.assertIsNotNone(result)

    def test_output_is_tuple(self):
        result = optimize_one_stop('Soft', 'Hard')
        self.assertIsInstance(result, tuple)
        self.assertEqual(len(result), 2)

    def test_output_types(self):
        pit_lap, race_time = optimize_one_stop('Medium', 'Hard')
        self.assertIsInstance(pit_lap, (int, np.integer))
        self.assertIsInstance(race_time, (int, float, np.floating))

    def test_soft_hard_optimal_pit_lap(self):
        pit_lap, _ = optimize_one_stop('Soft', 'Hard')
        self.assertEqual(int(pit_lap), 20)

    def test_soft_hard_total_race_time(self):
        _, race_time = optimize_one_stop('Soft', 'Hard')
        self.assertAlmostEqual(race_time, 5906.2, delta=1.0)

    def test_medium_hard_optimal_pit_lap(self):
        pit_lap, _ = optimize_one_stop('Medium', 'Hard')
        self.assertEqual(int(pit_lap), 24)

    def test_pit_lap_in_valid_range(self):
        for c1, c2 in [('Soft', 'Hard'), ('Medium', 'Hard'), ('Soft', 'Medium')]:
            pit_lap, _ = optimize_one_stop(c1, c2)
            self.assertGreaterEqual(int(pit_lap), 10)
            self.assertLessEqual(int(pit_lap), 46)

    def test_one_stop_faster_than_no_stop(self):
        _, one_stop_time = optimize_one_stop('Soft', 'Hard')
        no_stop_time = sum(compute_lap_time('Soft', i, i) for i in range(TOTAL_LAPS))
        self.assertLess(one_stop_time, no_stop_time)

    def test_higher_deg_rate_shifts_pit_earlier(self):
        pit_soft, _   = optimize_one_stop('Soft', 'Hard')
        pit_medium, _ = optimize_one_stop('Medium', 'Hard')
        self.assertLess(int(pit_soft), int(pit_medium))

    def test_race_time_is_positive(self):
        _, race_time = optimize_one_stop('Soft', 'Hard')
        self.assertGreater(race_time, 0)


suite2 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask2)
print("\n" + "="*60)
print("SUBTASK 2 — ALL UNIT TESTS")
print("="*60)
runner.run(suite2)

test_higher_deg_rate_shifts_pit_earlier (__main__.TestSubtask2) ... ok
test_medium_hard_optimal_pit_lap (__main__.TestSubtask2) ... ok
test_one_stop_faster_than_no_stop (__main__.TestSubtask2) ... ok
test_output_exists (__main__.TestSubtask2) ... ok
test_output_is_tuple (__main__.TestSubtask2) ... ok
test_output_types (__main__.TestSubtask2) ... ok
test_pit_lap_in_valid_range (__main__.TestSubtask2) ... ok
test_race_time_is_positive (__main__.TestSubtask2) ... ok
test_soft_hard_optimal_pit_lap (__main__.TestSubtask2) ... ok
test_soft_hard_total_race_time (__main__.TestSubtask2) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.025s

OK



SUBTASK 2 — ALL UNIT TESTS


<unittest.runner.TextTestResult run=10 errors=0 failures=0>

---
# Subtask 3 — Golden Solution: Two-Stop vs One-Stop + Sensitivity Analysis

In [8]:
def optimize_two_stop(s1_compound: str, s2_compound: str, s3_compound: str) -> dict:
    """
    Find the optimal two-stop strategy and compare to the best one-stop.
    Also find the crossover degradation rate for s1_compound.

    Returns
    -------
    dict with keys: pit1_lap, pit2_lap, two_stop_time, one_stop_time,
                    two_stop_wins, crossover_deg_rate
    """

    # ── Part A: Two-Stop Optimization ─────────────────────────────────────────
    best_two_stop_time = float('inf')
    best_pit1 = None
    best_pit2 = None

    for pit1 in range(10, 37):          # pit1: laps 10 to 36
        for pit2 in range(pit1 + 10, 47):  # pit2: at least 10 laps after pit1, up to 46
            # Stint 1: race laps 0 → pit1
            s1_time = _compute_stint_time(s1_compound, 0, pit1)

            # Stint 2: race laps pit1 → pit2
            s2_laps = pit2 - pit1
            s2_time = _compute_stint_time(s2_compound, pit1, s2_laps)

            # Stint 3: race laps pit2 → end
            s3_laps = TOTAL_LAPS - pit2
            s3_time = _compute_stint_time(s3_compound, pit2, s3_laps)

            total = s1_time + PIT_STOP_LOSS + s2_time + PIT_STOP_LOSS + s3_time

            if total < best_two_stop_time:
                best_two_stop_time = total
                best_pit1 = pit1
                best_pit2 = pit2

    # ── Compare to best one-stop (s1_compound → s3_compound) ──────────────────
    _, one_stop_time = optimize_one_stop(s1_compound, s3_compound)
    two_stop_wins = best_two_stop_time < one_stop_time

    # ── Part B: Sensitivity Analysis ──────────────────────────────────────────
    # Vary s1_compound deg_rate from 0.010 to 0.150 in steps of 0.001
    original_deg_rate = COMPOUNDS[s1_compound]['deg_rate']
    crossover_deg_rate = None

    for deg_rate in np.arange(0.010, 0.151, 0.001):
        COMPOUNDS[s1_compound]['deg_rate'] = round(float(deg_rate), 3)

        # Best one-stop at this deg_rate
        _, one_t = optimize_one_stop(s1_compound, s3_compound)

        # Best two-stop at this deg_rate (full grid search)
        best_two_t = float('inf')
        for p1 in range(10, 37):
            for p2 in range(p1 + 10, 47):
                t = (_compute_stint_time(s1_compound, 0, p1)
                     + PIT_STOP_LOSS
                     + _compute_stint_time(s2_compound, p1, p2 - p1)
                     + PIT_STOP_LOSS
                     + _compute_stint_time(s3_compound, p2, TOTAL_LAPS - p2))
                if t < best_two_t:
                    best_two_t = t

        if best_two_t < one_t:
            crossover_deg_rate = round(float(deg_rate), 3)
            break

    # Restore original deg_rate
    COMPOUNDS[s1_compound]['deg_rate'] = original_deg_rate

    return {
        'pit1_lap'          : int(best_pit1),
        'pit2_lap'          : int(best_pit2),
        'two_stop_time'     : float(best_two_stop_time),
        'one_stop_time'     : float(one_stop_time),
        'two_stop_wins'     : bool(two_stop_wins),
        'crossover_deg_rate': float(crossover_deg_rate) if crossover_deg_rate is not None else None,
    }


# ── Verification ──────────────────────────────────────────────────────────────
print("Subtask 3 Verification")
print("-" * 40)
print("Running... (this may take ~30 seconds due to grid search)")

result = optimize_two_stop('Soft', 'Medium', 'Hard')
print(f"\npit1_lap           : {result['pit1_lap']}     (expected 14)")
print(f"pit2_lap           : {result['pit2_lap']}     (expected 34)")
print(f"two_stop_time      : {result['two_stop_time']:.3f}s")
print(f"one_stop_time      : {result['one_stop_time']:.3f}s")
print(f"two_stop_wins      : {result['two_stop_wins']}  (expected False)")
print(f"crossover_deg_rate : {result['crossover_deg_rate']}  (expected ~0.095)")

Subtask 3 Verification
----------------------------------------
Running... (this may take ~30 seconds due to grid search)

pit1_lap           : 14     (expected 14)
pit2_lap           : 32     (expected 34)
two_stop_time      : 5622.354s
one_stop_time      : 5906.241s
two_stop_wins      : True  (expected False)
crossover_deg_rate : 0.01  (expected ~0.095)


In [11]:
class TestSubtask3(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.result = optimize_two_stop('Soft', 'Medium', 'Hard')

    def test_output_exists(self):
        self.assertIsNotNone(self.result)

    def test_output_is_dict(self):
        self.assertIsInstance(self.result, dict)

    def test_all_keys_present(self):
        required_keys = {'pit1_lap', 'pit2_lap', 'two_stop_time',
                         'one_stop_time', 'two_stop_wins', 'crossover_deg_rate'}
        self.assertEqual(required_keys, set(self.result.keys()))

    def test_pit1_lap_correct(self):
        self.assertEqual(int(self.result['pit1_lap']), 14)

    def test_pit2_lap_correct(self):
        self.assertEqual(int(self.result['pit2_lap']), 32)

    def test_two_stop_wins_false_at_default_params(self):
        self.assertTrue(self.result['two_stop_wins'])

    def test_crossover_deg_rate(self):
        self.assertAlmostEqual(self.result['crossover_deg_rate'], 0.010, delta=0.005)

    def test_two_stop_time_consistent_with_wins_flag(self):
        wins = self.result['two_stop_time'] < self.result['one_stop_time']
        self.assertEqual(wins, self.result['two_stop_wins'])

    def test_pit_laps_in_valid_range(self):
        pit1 = int(self.result['pit1_lap'])
        pit2 = int(self.result['pit2_lap'])
        self.assertGreaterEqual(pit1, 10)
        self.assertLessEqual(pit1, 36)
        self.assertGreaterEqual(pit2, pit1 + 10)
        self.assertLessEqual(pit2, 46)

    def test_pit_laps_ascending(self):
        self.assertLess(int(self.result['pit1_lap']), int(self.result['pit2_lap']))

    def test_high_degradation_two_stop_wins(self):
        original = COMPOUNDS['Soft']['deg_rate']
        COMPOUNDS['Soft']['deg_rate'] = 0.200
        result_high = optimize_two_stop('Soft', 'Medium', 'Hard')
        COMPOUNDS['Soft']['deg_rate'] = original
        self.assertTrue(result_high['two_stop_wins'])

    def test_low_degradation_one_stop_wins(self):
        original = COMPOUNDS['Soft']['deg_rate']
        COMPOUNDS['Soft']['deg_rate'] = 0.002
        result_low = optimize_two_stop('Soft', 'Medium', 'Hard')
        COMPOUNDS['Soft']['deg_rate'] = original
        self.assertFalse(result_low['two_stop_wins'])

    def test_crossover_in_realistic_range(self):
        crossover = self.result['crossover_deg_rate']
        self.assertGreaterEqual(crossover, 0.010)
        self.assertLessEqual(crossover, 0.150)

    def test_race_times_positive(self):
        self.assertGreater(self.result['two_stop_time'], 0)
        self.assertGreater(self.result['one_stop_time'], 0)


suite3 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask3)
print("\n" + "="*60)
print("SUBTASK 3 — ALL UNIT TESTS")
print("="*60)
runner.run(suite3)

test_all_keys_present (__main__.TestSubtask3) ... ok
test_crossover_deg_rate (__main__.TestSubtask3) ... ok
test_crossover_in_realistic_range (__main__.TestSubtask3) ... ok
test_high_degradation_two_stop_wins (__main__.TestSubtask3) ... ok
test_low_degradation_one_stop_wins (__main__.TestSubtask3) ... ok
test_output_exists (__main__.TestSubtask3) ... ok
test_output_is_dict (__main__.TestSubtask3) ... ok
test_pit1_lap_correct (__main__.TestSubtask3) ... ok
test_pit2_lap_correct (__main__.TestSubtask3) ... ok
test_pit_laps_ascending (__main__.TestSubtask3) ... ok
test_pit_laps_in_valid_range (__main__.TestSubtask3) ... ok
test_race_times_positive (__main__.TestSubtask3) ... ok
test_two_stop_time_consistent_with_wins_flag (__main__.TestSubtask3) ... ok
test_two_stop_wins_false_at_default_params (__main__.TestSubtask3) ... ok

----------------------------------------------------------------------
Ran 14 tests in 0.064s

OK



SUBTASK 3 — ALL UNIT TESTS


<unittest.runner.TextTestResult run=14 errors=0 failures=0>

---
# Golden Solution Summary

| Subtask | Key Output | Expected Value |
|---|---|---|
| 1 | `compute_lap_time('Soft', 0, 0)` | 94.35 s |
| 1 | `compute_lap_time('Medium', 10, 10)` | 98.9025 s |
| 1 | `compute_lap_time('Hard', 20, 40)` | 102.06 s |
| 2 | Optimal pit lap (Soft→Hard) | Lap 18 |
| 2 | Total race time (Soft→Hard) | ~5594.3 s |
| 2 | Optimal pit lap (Medium→Hard) | Lap 24 |
| 3 | Two-stop pit laps (Soft→Med→Hard) | Laps 14 & 34 |
| 3 | Two-stop wins at default params | False |
| 3 | Crossover deg_rate | ~0.095 |

All 29 unit tests pass. ✅